# 고객 소비 패턴 분석

`cc_num`, `category`, `amt`를 기준으로 고객별 소비 성향을 분석합니다.

확인할 내용:

- 고객별/카테고리별 거래횟수
- 고객별/카테고리별 총소비금액
- 고객별/카테고리별 평균소비금액
- 상위 소비 고객 10명의 주요 소비 카테고리
- 카테고리별 전체 소비 금액

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
from matplotlib import font_manager

In [ ]:
# 한글 폰트 설정
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

display(HTML("""
<style>
.jp-Notebook { font-size: 13px; }
.jp-RenderedMarkdown { font-size: 13px; line-height: 1.55; }
.jp-RenderedMarkdown h1 { font-size: 22px; }
.jp-RenderedMarkdown h2 { font-size: 17px; }
.jp-RenderedMarkdown h3 { font-size: 15px; }
.jp-OutputArea-output, .jp-RenderedText, .jp-OutputArea pre { font-size: 12px; }
.dataframe { font-size: 11px; }
</style>
"""))

## 1. 데이터 로드

현재 프로젝트에는 `data/credit_card_transactions.csv`가 없을 수 있으므로, 없으면 `step1/data/credit_card_transactions.csv`를 자동으로 사용합니다.

In [ ]:
def get_data_path():
    current_dir = Path.cwd()
    candidates = [
        current_dir / "data" / "credit_card_transactions.csv",
        current_dir / "step1" / "data" / "credit_card_transactions.csv",
        current_dir.parent / "data" / "credit_card_transactions.csv",
        current_dir.parent / "step1" / "data" / "credit_card_transactions.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "credit_card_transactions.csv 파일을 찾을 수 없습니다. "
        "data/ 또는 step1/data/ 폴더를 확인하세요."
    )


data_path = get_data_path()
print(f"[데이터 파일] {data_path}")

df = pd.read_csv(data_path)

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

required_columns = {"cc_num", "category", "amt"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing_columns)}")

print(df.shape)
df.head()

## 2. 고객별/카테고리별 소비 패턴 집계

고객 번호(`cc_num`)와 소비 카테고리(`category`)를 함께 묶어서, 고객이 어떤 영역에 자주 쓰고 많이 쓰는지 확인합니다.

In [ ]:
customer_category = (
    df.groupby(["cc_num", "category"])
    .agg(
        거래횟수=("amt", "count"),
        총소비금액=("amt", "sum"),
        평균소비금액=("amt", "mean"),
    )
    .reset_index()
)

customer_category.sort_values("총소비금액", ascending=False).head(20)

## 3. 고객별 대표 소비 카테고리

- `최다거래카테고리`: 가장 자주 결제한 카테고리
- `최대소비카테고리`: 가장 많은 금액을 쓴 카테고리

In [ ]:
top_category_by_count = (
    customer_category.sort_values(
        ["cc_num", "거래횟수", "총소비금액"],
        ascending=[True, False, False],
    )
    .drop_duplicates("cc_num")
    .set_index("cc_num")
)

top_category_by_amount = (
    customer_category.sort_values(
        ["cc_num", "총소비금액", "거래횟수"],
        ascending=[True, False, False],
    )
    .drop_duplicates("cc_num")
    .set_index("cc_num")
)

customer_summary = df.groupby("cc_num").agg(
    총소비금액=("amt", "sum"),
    평균소비금액=("amt", "mean"),
    거래횟수=("amt", "count"),
)

top_spenders = customer_summary.sort_values("총소비금액", ascending=False).head(10)
top_spender_patterns = top_spenders.copy()
top_spender_patterns["최다거래카테고리"] = top_category_by_count.loc[
    top_spender_patterns.index, "category"
]
top_spender_patterns["최다거래카테고리횟수"] = top_category_by_count.loc[
    top_spender_patterns.index, "거래횟수"
]
top_spender_patterns["최대소비카테고리"] = top_category_by_amount.loc[
    top_spender_patterns.index, "category"
]
top_spender_patterns["최대소비카테고리금액"] = top_category_by_amount.loc[
    top_spender_patterns.index, "총소비금액"
]

top_spender_patterns

## 4. 카테고리별 전체 소비 금액

전체 고객 기준으로 어떤 카테고리에 가장 많은 돈이 쓰였는지 확인합니다.

In [ ]:
category_spend = df.groupby("category")["amt"].sum().sort_values(ascending=False)
category_spend

In [ ]:
plt.figure(figsize=(10, 5))
category_spend.plot(kind="bar")
plt.title("카테고리별 전체 소비 금액")
plt.xlabel("카테고리")
plt.ylabel("총 소비 금액")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. 결과 해석

- 거래횟수가 가장 많은 카테고리: 고객이 자주 이용하는 소비 영역
- 총소비금액이 가장 큰 카테고리: 고객이 가장 많은 돈을 쓰는 영역
- 평균소비금액이 큰 카테고리: 한 번 결제할 때 금액이 큰 소비 영역

예시 해석:

- `grocery_pos` 비중이 높으면 식료품/생활 소비 중심 고객으로 볼 수 있습니다.
- `travel` 또는 `shopping_net` 금액이 크면 여행/온라인 쇼핑 중심 고객으로 볼 수 있습니다.
- 특정 카테고리에 거래가 몰려 있으면 마케팅 타겟 분류에 활용할 수 있습니다.